# Model #3 Tag-Visible Detector (Colab)

Run this notebook on Colab with a GPU runtime. It uses the fine-tuned Qwen LoRA adapter on Hugging Face to answer a simple zero-shot question: whether a clothing tag, label, or price tag is visible in the image.

Before running the full pass, put these large local data files in Google Drive, for example under `MyDrive/resell_copilot_data/`:

- `vinted_clothing_combined.parquet`
- `kleinanzeigen_clothing_combined.parquet`

The split JSON files are expected to come from the GitHub repo.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone Repo

This assumes the latest Model #3 code has been pushed to GitHub. If `models/run_tag_detector.py` is missing after cloning, push the local repo changes first, then rerun this cell.

In [ ]:
![ -d /content/Advanced_ML/.git ] || git clone https://github.com/mchlkan/Advanced_ML.git /content/Advanced_ML
%cd /content/Advanced_ML
!git pull --ff-only
!test -f models/run_tag_detector.py && echo 'Model #3 script found.' || echo 'ERROR: models/run_tag_detector.py is missing. Push the latest repo changes first.'

## 3. Install Dependencies

`--load-in-4bit` needs `bitsandbytes`, and the adapter needs `peft`.

In [ ]:
!pip -q install -U transformers accelerate peft bitsandbytes huggingface_hub pandas pyarrow pillow tqdm

## 4. Hugging Face Login

You already accepted the model terms. Paste a Hugging Face access token when prompted. A read token is enough.

In [ ]:
from huggingface_hub import login
login()

## 5. Copy Data From Google Drive

Change `DATA_SOURCE_DIR` if your Drive folder is different.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_SOURCE_DIR = '/content/drive/MyDrive/resell_copilot_data'

!mkdir -p data data/features
!cp "{DATA_SOURCE_DIR}/vinted_clothing_combined.parquet" data/vinted_clothing_combined.parquet
!cp "{DATA_SOURCE_DIR}/kleinanzeigen_clothing_combined.parquet" data/kleinanzeigen_clothing_combined.parquet
!ls -lh data/vinted_clothing_combined.parquet data/kleinanzeigen_clothing_combined.parquet data/splits/*.json

## 6. Validate Data Shape

In [ ]:
import json
import pandas as pd

vinted = pd.read_parquet('data/vinted_clothing_combined.parquet')
ka = pd.read_parquet('data/kleinanzeigen_clothing_combined.parquet')
print('vinted:', vinted.shape)
print('ka:', ka.shape)
for split in ['train', 'val', 'test']:
    ids = json.load(open(f'data/splits/{split}_ids.json'))
    print(split, len(ids))

## 7. Smoke Test On 10 Rows

This downloads the base model + adapter and runs only 10 examples. On a T4, model loading is the slow part.

In [ ]:
!python models/run_tag_detector.py \
  --model-id Qwen/Qwen3-VL-4B-Instruct \
  --adapter-id Rengo33/qwen3vl4b-resell-adapter \
  --load-in-4bit \
  --output /content/tag_visible_smoke.parquet \
  --limit 10 \
  --batch-size 1

## 8. Inspect Smoke Test

In [ ]:
smoke = pd.read_parquet('/content/tag_visible_smoke.parquet')
smoke

If `tag_visible` is mostly `0` or `1` and `raw_response` is mostly `yes` or `no`, run the full dataset below.

## 9. Run Full Dataset

In [ ]:
!python models/run_tag_detector.py \
  --model-id Qwen/Qwen3-VL-4B-Instruct \
  --adapter-id Rengo33/qwen3vl4b-resell-adapter \
  --load-in-4bit \
  --output data/features/tag_visible_combined.parquet \
  --batch-size 1

## 10. Validate And Save Output

In [ ]:
out = pd.read_parquet('data/features/tag_visible_combined.parquet')
print(out.shape)
print(out['tag_visible'].value_counts(dropna=False))
print('duplicates:', out.duplicated(['platform', 'id']).sum())
print(out.head().to_string())

# Keep a copy in Drive so it survives Colab runtime reset.
!cp data/features/tag_visible_combined.parquet "{DATA_SOURCE_DIR}/tag_visible_combined.parquet"
print('Copied to:', DATA_SOURCE_DIR + '/tag_visible_combined.parquet')

## 11. Optional Download To Local Machine

In [ ]:
from google.colab import files
files.download('data/features/tag_visible_combined.parquet')